# Evaluate by-example WSD with DeBERTa-Chinese


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
MODEL_NAME = "Deberta-Chinese-Large"
NB_ID = "23.66"

In [ ]:
import json
import sys

sys.path.append("./dotted_wsd_for_debertachinese")

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
)
from tqdm.auto import tqdm

## Data Hash

```
../data/WSD_examples  : b9bbaf
```

In [ ]:
wsd_examples_path = "./dotted-wsd/data/wsd_examples.csv"

CHECK_DATA_HASH = True
if CHECK_DATA_HASH:
    import hashlib
    from pathlib import Path

    for data_path in (wsd_examples_path,):
        hasher = hashlib.sha1()
        hasher.update(Path(data_path).read_bytes())
        h = hasher.digest().hex()[:6]
        print(f"{data_path:<30s}: {h}")

In [ ]:
wsd_examples = pd.read_csv(wsd_examples_path, index_col=0)

In [ ]:
wsd_examples.shape

In [ ]:
wsd_examples.head()

## Evaluation

#### Using `wsd_examples.csv` 9152

**OLD**: The evaluation accuracy computed with 15.22 is **.8619** (with POS hint)






```
wsd_eval = evaluate(model, wsd_evalloader)
```

In [ ]:
from dotted_wsd_for_debertachinese import DottedWsdTagger

DeBERTatagger = DottedWsdTagger()

In [ ]:
input_text = wsd_examples.iloc[0].test_sentence
pos_hint = wsd_examples.iloc[0].test_pos
ref_sense_id = wsd_examples.iloc[0].test_sense_id

input_text, pos_hint, ref_sense_id

In [ ]:
ex_pred, inst_preds = DeBERTatagger.wsd_tag(input_text, pos_hint)
print(ex_pred.prediction())
print("predict class (sense_id): ", ex_pred.pred_class, "predict prob: ", ex_pred.prob)
print(len(inst_preds))

## With POS hints

In [ ]:
predictions = []
for _, row in tqdm(wsd_examples.iterrows(), total=wsd_examples.shape[0]):
    try:
        input_text = row.test_sentence
        pos_hint = row.test_pos
        ex_pred, inst_preds = DeBERTatagger.wsd_tag(input_text, pos_hint)
        predictions.append((ex_pred.pred_class, ex_pred.prob, len(inst_preds)))
    except:
        predictions.append(("----", 0.0, len(inst_preds)))

In [ ]:
pred_poshint = [x[0] for x in predictions]
prob_poshint = [x[1] for x in predictions]
ncandid_poshint = [x[2] for x in predictions]

In [ ]:
ref_labels = wsd_examples.test_sense_id
acc_pos_hint = accuracy_score(ref_labels, pred_poshint)
acc_pos_hint

## NO POS hint

In [ ]:
predictions = []
for _, row in tqdm(wsd_examples.iterrows(), total=wsd_examples.shape[0]):
    try:
        input_text = row.test_sentence
        ex_pred, inst_preds = DeBERTatagger.wsd_tag(input_text)
        predictions.append((ex_pred.pred_class, ex_pred.prob, len(inst_preds)))
    except:
        predictions.append(("----", 0.0, len(inst_preds)))

In [ ]:
pred_noposhint = [x[0] for x in predictions]
prob_noposhint = [x[1] for x in predictions]
ncandid_noposhint = [x[2] for x in predictions]

In [ ]:
ref_labels = wsd_examples.test_sense_id
acc_nopos_hint = accuracy_score(ref_labels, pred_noposhint)
acc_nopos_hint

## Outputs

In [ ]:
wsd_examples_evals = wsd_examples.assign(
    pred_poshint=pred_poshint,
    prob_poshint=prob_poshint,
    ncandid_poshint=ncandid_poshint,
    pred_noposhint=pred_noposhint,
    prob_noposhint=prob_noposhint,
    ncandid_noposhint=ncandid_noposhint,
)
wsd_examples_evals_path = f"./data/{MODEL_NAME}_{NB_ID}_wsd_example_evals.csv"
wsd_examples_evals.to_csv(wsd_examples_evals_path, index=False)

In [ ]:
h = hashlib.sha1()
h.update(Path(wsd_examples_evals_path).read_bytes())
print(wsd_examples_evals_path, h.hexdigest()[:6])

## Metrics

In [ ]:
out_metric_path = f"./data/metrics/{MODEL_NAME}_wsd-eval-by-example-metrics.json"
with open(out_metric_path, "w") as fout:
    json.dump(
        {
            "acc_pos_hint": acc_pos_hint,
            "acc_nopos_hint": acc_nopos_hint,
        },
        fout,
    )

In [ ]:
h = hashlib.sha1()
h.update(Path(out_metric_path).read_bytes())
print(out_metric_path, h.hexdigest()[:6])